In [1]:
# debug cell (restart kernel and run before running optimizer)
import sys
import os
import math

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    AspectRatio,
    ForceBalance,
    QuasisymmetryBoozer,
    QuasisymmetryTwoTerm,
    QuasisymmetryTripleProduct,
    ObjectiveFromUser,
    GammaC,
    TrappedResonance
)
from desc.optimize import Optimizer
from desc.plotting import (
    plot_grid,
    plot_boozer_modes,
    plot_boozer_surface,
    plot_qs_error,
    plot_boundaries,
    plot_boundary,
)
# load initial equilibrium
# eq_init = desc.io.load("qs_initial_guess.h5") # QI
eq_init = desc.io.load("Equilibria/desc_eq_beta2.5_QA.h5") # QA

# Specify equilibrium nfp and helicity
# N = -1 # QH helicity
N_h = 0 # QA
nfp = eq_init.NFP
s_input = np.linspace(0.1,0.9,10) # surfaces to test, rho = sqrt(s) is calculated below. DESC 

# optimizer = Optimizer("proximal-scipy-bfgs")
optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
# idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
# idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
# idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
# R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
# Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc], axis=0)
Z_modes = eq_init.surface.Z_basis.modes

eq_qs_T = eq_init.copy()  # make a copy of the original one
# constraints
constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixIota(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# Create a grid in (rho, theta, zeta) coordinates
# rho = np.linspace(0.1,0.9,5)
rho=abs(np.sqrt(s_input))
eq_periodicity = (np.inf,np.inf,np.inf) # periodicity in zeta for these equilibrium to make rtz grid
alphas=np.linspace(0,2*np.pi,2)
grid = eq_init._get_rtz_grid( # returns rho, theta, zeta coordinate grid
    rho, # radial
    alphas, # poloidal (alpha in this case)
    np.linspace(0, 9 * np.pi, 50), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
# grid has the number of nodes equal to len(rho)*len(alpha)*length(zeta)

# Grid for QuasisymmetryTripleProduct objective
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)

grid_psi = eq_init._get_rtz_grid( # returns rho, theta, zeta coordinate grid
    rho, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.array([0]), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
Psi = eq_init.compute("Psi",grid=grid_psi)
# obj = TrappedResonance(eq_init,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi'])

import jax.numpy as jnp
pitch_invs = jnp.linspace(5.8,6.7,5)


# Objective for resonance
objective_fT = ObjectiveFunction(
    (
        QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol),
        TrappedResonance(eq=eq_qs_T, grid=grid, N=N_h,Psi=Psi['Psi'],rho=rho,pitch_invs=pitch_invs,KE_frac=np.array([1]),alpha=alphas)
    ), 
)

surface.R_basis.modes is an array of [l,m,n] of the surface modes:
[[  0 -15 -12]
 [  0 -14 -12]
 [  0 -13 -12]
 [  0 -12 -12]
 [  0 -11 -12]
 [  0 -10 -12]
 [  0  -9 -12]
 [  0  -8 -12]
 [  0  -7 -12]
 [  0  -6 -12]]


In [2]:
eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=10,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

Building objective: QS triple product
Precomputing transforms
Timer: Precomputing transforms = 573 ms
Building objective: TrappedResonance
Precomputing transforms
Timer: Precomputing transforms = 662 ms
Timer: Objective build = 2.30 sec
Building objective: force
Precomputing transforms
Timer: Precomputing transforms = 324 ms
Timer: Objective build = 355 ms
Timer: Objective build = 1.20 ms
Timer: Eq Update LinearConstraintProjection build = 4.70 sec
Timer: Proximal projection build = 11.3 sec
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed pressure
Building objective: fixed iota
Building objective: fixed Psi
Timer: Objective build = 290 ms
Timer: LinearConstraintProjection build = 1.32 sec
Number of parameters: 1
Number of objectives: 12554
Timer: Initializing the optimization = 13.0 sec

Starting optimization
Using method: proximal-lsq-exact


TypeError: 'NoneType' object is not subscriptable

In [6]:
# debug cell (restart kernel and run before running optimizer)
import sys
import os
import math

# sys.path.insert(0, os.path.abspath("."))
# sys.path.append(os.path.abspath("../../../"))
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    AspectRatio,
    ForceBalance,
    QuasisymmetryBoozer,
    QuasisymmetryTwoTerm,
    QuasisymmetryTripleProduct,
    ObjectiveFromUser,
    GammaC,
    TrappedResonance
)
from desc.optimize import Optimizer
from desc.plotting import (
    plot_grid,
    plot_boozer_modes,
    plot_boozer_surface,
    plot_qs_error,
    plot_boundaries,
    plot_boundary,
)
# load initial equilibrium
# eq_init = desc.io.load("qs_initial_guess.h5") # QI
eq_init = desc.io.load("Equilibria/desc_eq_beta2.5_QA.h5") # QA

# Specify equilibrium nfp and helicity
# N = -1 # QH helicity
N_h = 0 # QA
nfp = eq_init.NFP


# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis

In [7]:
print(idx_Rcc)
print(idx_Rcc)

DoubleFourierSeries at 0x38e1ab5b0 (L=0, M=15, N=12, NFP=2, sym=cos, spectral_indexing=linear)


: 